# Data Processing for NSW Road Crash Analysis

This notebook cleans and merges the two dataset NSW Crash Dataset and ABS population dataset from the raw folder. The data are processed and stored in the processed folder as two separate datasets, one for clustering and one for classification.

* Inputs: `the NSW crash dataset's Sheet2` and `LGA_population.xlsx`
* Outputs: `lga_yearly_aggregates_clustering.csv` & `nsw_crashes_classification.csv`

Note: Refer to the README.md file for more detailed explanation


## Cell 1: Importing Pandas

In [1]:
import pandas as pd

## Cell 2: Reading the Crash Dataset

In [2]:
crash_data = pd.read_excel('../data/raw/nsw_road_crash_data_2020-2024_crash.xlsx', sheet_name="Sheet2")
print(crash_data.shape)

(92189, 12)


## Cell 3: Reading the Population Dataset

In [3]:
population_data = pd.read_excel('../data/raw/LGA_population.xlsx')
population_data.shape

(546, 7)

## Cell 4: Finding LGA name mismatches

In [4]:
# Get unique values and convert them to sets
crash_lgas = set(crash_data["LGA"].unique())
population_lgas = set(population_data["LGA name"].unique())

# Subtract population LGAs from crash LGAs to find mismatches
mismatches = crash_lgas - population_lgas

print(f"Found {len(mismatches)} mismatched LGA names:")
print(mismatches)


Found 12 mismatched LGA names:
{'Lord Howe Island', 'Bathurst Regional', 'Tamworth Regional', 'Mid-Western Regional', 'Dubbo Regional', 'Unincorporated', 'Bayside', 'Queanbeyan-Palerang Regional', 'Snowy Monaro Regional', 'Armidale Regional', 'Campbelltown', 'Central Coast'}


## Cell 5: Renaming the LGA and dropping Lord Howe Island

In [5]:
# Create a new dictionary with the replacement names
lga_rename_dictionary = {"Armidale Regional": "Armidale", "Dubbo Regional": "Dubbo","Queanbeyan-Palerang Regional": "Queanbeyan-Palerang", 
            "Snowy Monaro Regional": "Snowy Monaro", "Campbelltown": "Campbelltown (NSW)", "Central Coast": "Central Coast (NSW)",
            "Tamworth Regional": "Tamworth", "Mid-Western Regional": "Mid-Western", "Bayside": "Bayside (NSW)",
            "Unincorporated": "Unincorporated NSW", "Bathurst Regional": "Bathurst"}

# Replace the crash dataset lga names
crash_data["LGA"] = crash_data["LGA"].replace(lga_rename_dictionary)

# Remove Lord Howe Island from crash dataset as the population dataset 
# do not have data for this LGA
crash_data = crash_data[crash_data["LGA"] != "Lord Howe Island"]

crash_lgas = set(crash_data["LGA"].unique())
population_lgas = set(population_data["LGA name"].unique())

# Check the mismatches again
new_mismatches = crash_lgas - population_lgas
print(new_mismatches)
print(crash_data.shape)


set()
(92184, 12)


## Cell 6: Filtering out 2019 crashes

In [6]:
# Filter to keep only 2020-2024 and make an explicit copy to follow the study window
# and 2019 has only tiny entries
crash_filtered = crash_data[crash_data["Year of crash"].isin([2020, 2021, 2022, 2023, 2024])].copy()

# Print the shape to check row counts
print("Crash Data Shape:", crash_filtered.shape)


Crash Data Shape: (92077, 12)


## Cell 7: Filtering out non-NSW LGAs

In [7]:
# Check the starting digit of LGA code.
# filter with "1" because NSW LGA IDs all begin with 1.
# Also ensure the LGA code column is treated as a string 
population_nsw = population_data[population_data["LGA code"].astype(str).str.startswith("1")].copy()

# Print the shape to check the rows
print("Population Data (NSW) Shape:", population_nsw.shape)


Population Data (NSW) Shape: (129, 7)


## Cell 8: Changing the format of Population data and Merging the two dataset

In [8]:
# Convert wide population table (one row per LGA) into 
# a long format(one row per LGA per year)for merging
population_melted = pd.melt(
    population_nsw, 
    id_vars=["LGA code", "LGA name"], 
    value_vars=[2020, 2021, 2022, 2023, 2024],
    var_name="Year", 
    value_name="Population"
)

# Convert Year to integer so it matches the crash data type
population_melted["Year"] = population_melted["Year"].astype(int)

# Verify the melted shape
print("Melted Population Shape:", population_melted.shape)

# Merge onto the filtered crash data
merged_data = pd.merge(
    crash_filtered, 
    population_melted, 
    left_on=["LGA", "Year of crash"], 
    right_on=["LGA name", "Year"], 
    how="left"
)

# Final verification checks
print("Merged Table Shape:", merged_data.shape)
print("Missing Population Count:", merged_data["Population"].isna().sum())
print("Final columns:", list(merged_data.columns))



Melted Population Shape: (645, 4)
Merged Table Shape: (92077, 16)
Missing Population Count: 0
Final columns: ['Crash ID', 'Degree of crash', 'Degree of crash - detailed', 'Year of crash', 'Month of crash', 'Weather', 'Road classification (admin)', 'Road surface', 'Surface condition', 'LGA', 'Urbanisation', 'Speed limit', 'LGA code', 'LGA name', 'Year', 'Population']


## Cell 9: Turning month of crash into season

In [9]:
# Drop the duplicate columns from the merge
merged_tidy = merged_data.drop(columns=["LGA name", "Year"])

# Create the Season mapping dictionary
season_map = {
    'December': 'Summer', 'January': 'Summer', 'February': 'Summer',
    'March': 'Autumn', 'April': 'Autumn', 'May': 'Autumn',
    'June': 'Winter', 'July': 'Winter', 'August': 'Winter',
    'September': 'Spring', 'October': 'Spring', 'November': 'Spring',
}

# Apply mapping to 'Month of crash'
merged_tidy["Season"] = merged_tidy["Month of crash"].map(season_map)

# Verification checks
print("Missing Seasons:", merged_tidy["Season"].isna().sum())
print("\nSeason Distribution:\n", merged_tidy["Season"].value_counts())
print("Tidy Table Shape:", merged_tidy.shape)



Missing Seasons: 0

Season Distribution:
 Season
Autumn    23734
Spring    22983
Winter    22963
Summer    22397
Name: count, dtype: int64
Tidy Table Shape: (92077, 15)


## Cell 10: Grouping casualty and non-casualty

In [10]:
# Create a 1 for casualty (Fatal/Injury) and 0 for non-casualty towaways
merged_tidy["Severe_Crash"] = (merged_tidy["Degree of crash"] != "Non-casualty (towaway)").astype(int)

# Verification check
print("Casualty Crash Distribution:")
print(merged_tidy["Severe_Crash"].value_counts())
print("\nPercentages:\n", merged_tidy["Severe_Crash"].value_counts(normalize=True) * 100)
print(merged_tidy.shape)


Casualty Crash Distribution:
Severe_Crash
1    62570
0    29507
Name: count, dtype: int64

Percentages:
 Severe_Crash
1    67.953995
0    32.046005
Name: proportion, dtype: float64
(92077, 16)


## Cell 11: Filtering out the "Unknown" rows and converting speed limit to integer

In [11]:
# Filter out the "Unknown" rows across certain columns because they are less than 2%
# in total and carry no useful information 
crash_cleaned = merged_tidy[
    (merged_tidy["Weather"] != "Unknown") &
    (merged_tidy["Surface condition"] != "Unknown") &
    (merged_tidy["Road surface"] != "Unknown") &
    (merged_tidy["Speed limit"] != "Unknown") &
    (merged_tidy["Urbanisation"] != "Country unknown")
].copy()

# Strip " km/h" string and convert Speed limit to numeric integer
crash_cleaned["Speed limit"] = crash_cleaned["Speed limit"].str.replace(" km/h", "", case=False).astype(int)

# Final verification checks
print("Final Cleaned Dataset Shape:", crash_cleaned.shape)
print("Speed Limit Data Type:", crash_cleaned["Speed limit"].dtypes)
print("\nSample of Speed Limits:\n", crash_cleaned["Speed limit"].value_counts())


Final Cleaned Dataset Shape: (90857, 16)
Speed Limit Data Type: int64

Sample of Speed Limits:
 Speed limit
50     30224
60     24920
100    10140
80      8777
70      7628
110     3811
40      3787
90      1278
30       190
10        64
20        38
Name: count, dtype: int64


## Cell 12: Cleaning Up for Classification 

In [12]:
# Select and drop columns not suited for final classification model features
drop_cols = ["Degree of crash", "Degree of crash - detailed", "Month of crash"]
classification_df = crash_cleaned.drop(columns=drop_cols, errors="ignore")

# Export to CSV
classification_filepath = "../data/processed/nsw_crashes_classification.csv"
classification_df.to_csv(classification_filepath, index=False)

print(f"Classification file saved to: {classification_filepath}")
print("Final columns:", list(classification_df.columns))
print("Final shape:", classification_df.shape)
classification_df.head(10)

Classification file saved to: ../data/processed/nsw_crashes_classification.csv
Final columns: ['Crash ID', 'Year of crash', 'Weather', 'Road classification (admin)', 'Road surface', 'Surface condition', 'LGA', 'Urbanisation', 'Speed limit', 'LGA code', 'Population', 'Season', 'Severe_Crash']
Final shape: (90857, 13)


,Crash ID,Year of crash,Weather,Road classification (admin),Road surface,Surface condition,LGA,Urbanisation,Speed limit,LGA code,Population,Season,Severe_Crash
0,1220987,2020,Fine,State,Sealed,Dry,Cootamundra-Gundagai,Country non-urban,110,12160,11393,Summer,1
1,1220988,2020,Fine,State,Sealed,Dry,Clarence Valley,Country non-urban,100,11730,52893,Summer,1
2,1221458,2020,Fine,State,Sealed,Dry,Upper Lachlan,Country non-urban,100,17640,8378,Summer,1
3,1221459,2020,Fine,State,Sealed,Dry,Ryde,Sydney metro. area,60,16700,130999,Summer,1
5,1222087,2020,Raining,Regional,Sealed,Wet,Port Stephens,Country non-urban,90,16400,74450,Summer,1
7,1222090,2020,Overcast,State,Sealed,Wet,Wollondilly,Country non-urban,110,18400,53618,Summer,1
8,1222341,2020,Fine,Local,Sealed,Dry,Cumberland,Sydney metro. area,50,12380,240841,Summer,1
9,1222628,2020,Fine,State,Sealed,Dry,Griffith,Country non-urban,100,13450,27249,Summer,1
10,1222629,2020,Fine,State,Sealed,Dry,Mid-Western,Country non-urban,100,15270,25589,Summer,1
11,1222630,2020,Fine,Local,Sealed,Dry,Bellingen,Country non-urban,100,10600,13152,Summer,1


## Cell 13: Cleaning Up for Clustering

In [13]:
# Create simple 1 and 0 flags for different crash types
# is_wet derived from Surface Condition column from crash dataset
# is_rural derived from Urbanisation column from crash dataset
# Note: 'Severe %' actually represents the casualty share (injury/fatal) per LGA
merged_tidy["is_casualty"] = (merged_tidy["Severe_Crash"] == 1).astype(int)
merged_tidy["is_non_severe"] = (merged_tidy["Severe_Crash"] == 0).astype(int)
merged_tidy["is_wet"] = (merged_tidy["Surface condition"] == "Wet").astype(int)
merged_tidy["is_rural"] = (merged_tidy["Urbanisation"] == "Country non-urban").astype(int)

# Group the data by LGA and Year only
crash_counts = merged_tidy.groupby(["LGA", "Year of crash"]).agg(
    total_crashes=("Crash ID", "count"),
    severe_count=("is_casualty", "sum"),
    non_severe_count=("is_non_severe", "sum"),
    wet_crashes=("is_wet", "sum"),
    rural_crashes=("is_rural", "sum")
).reset_index()

# Merge onto the population table so LGA with 0 crashes don't disappear
clustering_df = pd.merge(
    population_melted,
    crash_counts,
    left_on=["LGA name", "Year"],
    right_on=["LGA", "Year of crash"],
    how="left"
)

# If an LGA had zero crashes, change the blank spaces (NaN) to 0
fill_cols = ["total_crashes", "severe_count", "non_severe_count", "wet_crashes", "rural_crashes"]
clustering_df[fill_cols] = clustering_df[fill_cols].fillna(0)

# Calculate the percentages and rates required for clustering
clustering_df["Severe %"] = (clustering_df["severe_count"] / clustering_df["total_crashes"]).fillna(0) * 100
clustering_df["Wet-weather %"] = (clustering_df["wet_crashes"] / clustering_df["total_crashes"]).fillna(0) * 100
clustering_df["Rural-road %"] = (clustering_df["rural_crashes"] / clustering_df["total_crashes"]).fillna(0) * 100
clustering_df["Crash rate per 100k"] = (clustering_df["total_crashes"] / clustering_df["Population"]) * 100000

# Clean up the names and organize the columns for clustering
clustering_df = clustering_df.drop(columns=["LGA name", "Year of crash"])
final_columns = [
    "LGA", "Year", "Population", "severe_count", "non_severe_count", 
    "Wet-weather %", "Rural-road %", "Crash rate per 100k", "Severe %", "total_crashes"
]
clustering_df = clustering_df[final_columns]

# Save clean table to a new CSV file
clustering_df.to_csv("../data/processed/lga_yearly_aggregates_clustering.csv", index=False)

# Verify the final clustering dataset 
print("Final Clustering Dataset Shape:", clustering_df.shape)
clustering_df.head(10)


Final Clustering Dataset Shape: (645, 10)


,LGA,Year,Population,severe_count,non_severe_count,Wet-weather %,Rural-road %,Crash rate per 100k,Severe %,total_crashes
0,Albury,2020,55466,73,50,12.195122,13.821138,221.757473,59.349593,123
1,Armidale,2020,29600,47,10,24.561404,56.140351,192.567568,82.456140,57
2,Ballina,2020,45663,58,19,22.077922,23.376623,168.626678,75.324675,77
3,Balranald,2020,2257,5,1,0.000000,83.333333,265.839610,83.333333,6
4,Bathurst,2020,43444,81,28,23.853211,32.110092,250.897707,74.311927,109
5,Bayside (NSW),2020,176184,276,122,14.321608,0.000000,225.900195,69.346734,398
6,Bega Valley,2020,35428,71,37,31.481481,50.000000,304.843627,65.740741,108
7,Bellingen,2020,13152,39,16,32.727273,34.545455,418.187348,70.909091,55
8,Berrigan,2020,8653,11,4,20.000000,60.000000,173.350283,73.333333,15
9,Blacktown,2020,391911,560,335,15.865922,0.000000,228.368175,62.569832,895
